# SFT on GSM8K — Answer Only

Train Llama-3.2-1B on GSM8K math problems using only the final numeric answer.

The model learns to produce the correct number without any reasoning trace.
Compare with `sft_gsm8k_cot.ipynb` to see what chain-of-thought training actually adds.

In [1]:
import re
import tinker
from tinker import types
from tinker.types import AdamParams
from datasets import load_dataset
from tinker_cookbook import renderers
from dotenv import load_dotenv
import numpy as np

load_dotenv()

True

In [2]:
# Load GSM8K
dataset = load_dataset("openai/gsm8k", "main", split="train")
test_dataset = load_dataset("openai/gsm8k", "main", split="test")

print(f"Train size: {len(dataset)}")
print(f"Test size: {len(test_dataset)}")
print("\nSample example:")
print("Q:", dataset[0]["question"])
print("A:", dataset[0]["answer"])

README.md: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Train size: 7473
Test size: 1319

Sample example:
Q: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
A: Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


In [3]:
def extract_answer(text: str) -> str | None:
    """Extract the final numeric answer after ####."""
    match = re.search(r'####\s*(-?[\d,]+)', text)
    return match.group(1).replace(',', '') if match else None

# Verify
sample = dataset[0]
print("Raw answer:", sample["answer"])
print("Extracted:", extract_answer(sample["answer"]))

Raw answer: Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72
Extracted: 72


In [4]:
service_client = tinker.ServiceClient()

training_client = service_client.create_lora_training_client(
    base_model="meta-llama/Llama-3.2-1B",
    rank=32
)

baseline_sampler = service_client.create_sampling_client(
    base_model="meta-llama/Llama-3.2-1B"
)

tokenizer = training_client.get_tokenizer()
renderer = renderers.get_renderer("llama3", tokenizer)

In [5]:
def build_messages(example: dict) -> list:
    return [
        {
            "role": "system",
            "content": "You are a math assistant. Answer with only the final numeric answer."
        },
        {
            "role": "user",
            "content": example["question"]
        },
        {
            "role": "assistant",
            # Just the number — no reasoning
            "content": extract_answer(example["answer"])
        }
    ]

def process_example(example: dict) -> types.Datum:
    messages = build_messages(example)
    model_input, weights = renderer.build_supervised_example(messages)
    tokens = model_input.tolist()
    weights = weights.tolist()
    return types.Datum(
        model_input=types.ModelInput.from_ints(tokens[:-1]),
        loss_fn_inputs={
            "target_tokens": tokens[1:],
            "weights": weights[1:]
        }
    )

# Preview one training example
print("Training example assistant turn:")
print(build_messages(dataset[0])[2]["content"])

Training example assistant turn:
72


In [6]:
N_TRAIN = 500

training_data = [
    process_example(ex)
    for ex in dataset.select(range(N_TRAIN))
]

print(f"Built {len(training_data)} training examples")

Built 500 training examples


In [7]:
N_STEPS = 50

for step in range(N_STEPS):
    fwd = training_client.forward_backward(
        training_data,
        loss_fn="cross_entropy"
    )
    opt = training_client.optim_step(
        AdamParams(learning_rate=1e-4)
    )

    fwd_res = fwd.result()
    opt.result()

    logprobs = np.concatenate(
        [o["logprobs"].tolist() for o in fwd_res.loss_fn_outputs]
    )
    weights = np.concatenate(
        [d.loss_fn_inputs["weights"].tolist() for d in training_data]
    )

    loss = -np.dot(logprobs, weights) / weights.sum()
    print(f"Step {step:02d} | Loss {loss:.4f}")

Step 00 | Loss 11.5681
Step 01 | Loss 10.4777
Step 02 | Loss 8.9786
Step 03 | Loss 7.7883
Step 04 | Loss 7.2324
Step 05 | Loss 6.8719
Step 06 | Loss 6.6086
Step 07 | Loss 6.4135
Step 08 | Loss 6.2493
Step 09 | Loss 6.0989
Step 10 | Loss 5.9738
Step 11 | Loss 5.8748
Step 12 | Loss 5.7899
Step 13 | Loss 5.6671
Step 14 | Loss 5.5599
Step 15 | Loss 5.4474
Step 16 | Loss 5.3576
Step 17 | Loss 5.2607
Step 18 | Loss 5.1590
Step 19 | Loss 5.0785
Step 20 | Loss 4.9871
Step 21 | Loss 4.9001
Step 22 | Loss 4.8139
Step 23 | Loss 4.7165
Step 24 | Loss 4.6256
Step 25 | Loss 4.5180
Step 26 | Loss 4.4107
Step 27 | Loss 4.2853
Step 28 | Loss 4.1517
Step 29 | Loss 4.0132
Step 30 | Loss 3.8732
Step 31 | Loss 3.7086
Step 32 | Loss 3.5456
Step 33 | Loss 3.3597
Step 34 | Loss 3.1594
Step 35 | Loss 2.9496
Step 36 | Loss 2.7347
Step 37 | Loss 2.4937
Step 38 | Loss 2.2994
Step 39 | Loss 2.1183
Step 40 | Loss 1.9042
Step 41 | Loss 1.5869
Step 42 | Loss 1.4534
Step 43 | Loss 1.1755
Step 44 | Loss 1.0351
Step 45 

In [8]:
sampler = training_client.save_weights_and_get_sampling_client("gsm8k-sft-answer-only")

In [9]:
# Evaluate on held-out test examples
N_EVAL = 100
eval_examples = test_dataset.select(range(N_EVAL))

stop_sequences = renderer.get_stop_sequences()
sampling_params = types.SamplingParams(
    max_tokens=32,   # answer-only needs very few tokens
    temperature=0.0,
    stop=stop_sequences
)

baseline_correct = 0
trained_correct = 0

for ex in eval_examples:
    prompt = renderer.build_generation_prompt([
        {
            "role": "system",
            "content": "You are a math assistant. Answer with only the final numeric answer."
        },
        {
            "role": "user",
            "content": ex["question"]
        }
    ])

    ground_truth = extract_answer(ex["answer"])

    def parse_answer_only(content: str) -> str:
        """For answer-only model, the whole output should be a number."""
        # try #### format first, fallback to stripping and using raw output
        ans = extract_answer(content)
        if ans:
            return ans
        return content.strip().replace(',', '')

    # Baseline
    baseline_res = baseline_sampler.sample(
        prompt=prompt, num_samples=1, sampling_params=sampling_params
    ).result()
    baseline_response, _ = renderer.parse_response(baseline_res.sequences[0].tokens)
    if parse_answer_only(baseline_response["content"]) == ground_truth:
        baseline_correct += 1

    # Trained
    trained_res = sampler.sample(
        prompt=prompt, num_samples=1, sampling_params=sampling_params
    ).result()
    trained_response, _ = renderer.parse_response(trained_res.sequences[0].tokens)
    if parse_answer_only(trained_response["content"]) == ground_truth:
        trained_correct += 1

print("=" * 50)
print(f"Eval on {N_EVAL} held-out GSM8K problems")
print("=" * 50)
print(f"Baseline accuracy       : {baseline_correct / N_EVAL * 100:.1f}%")
print(f"SFT (answer only) acc   : {trained_correct / N_EVAL * 100:.1f}%")
print(f"Delta                   : +{(trained_correct - baseline_correct) / N_EVAL * 100:.1f}pp")
print("=" * 50)

Eval on 100 held-out GSM8K problems
Baseline accuracy       : 0.0%
SFT (answer only) acc   : 6.0%
Delta                   : +6.0pp


In [10]:
# Qualitative comparison on a single example
ex = test_dataset[0]

prompt = renderer.build_generation_prompt([
    {
        "role": "system",
        "content": "You are a math assistant. Answer with only the final numeric answer."
    },
    {
        "role": "user",
        "content": ex["question"]
    }
])

print("QUESTION:", ex["question"])
print("=" * 50)

baseline_res = baseline_sampler.sample(
    prompt=prompt, num_samples=1,
    sampling_params=types.SamplingParams(max_tokens=32, temperature=0.0, stop=stop_sequences)
).result()
baseline_response, _ = renderer.parse_response(baseline_res.sequences[0].tokens)
print("BASELINE output  :", baseline_response["content"])

trained_res = sampler.sample(
    prompt=prompt, num_samples=1,
    sampling_params=types.SamplingParams(max_tokens=32, temperature=0.0, stop=stop_sequences)
).result()
trained_response, _ = renderer.parse_response(trained_res.sequences[0].tokens)
print("SFT (ans only)   :", trained_response["content"])

print("Ground truth     :", extract_answer(ex["answer"]))

QUESTION: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
BASELINE output  : Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She
SFT (ans only)   : 19
Ground truth     : 18
